In [10]:
import os
from load_dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnableLambda

if os.environ.get("OPENAI_API_KEY"):
  print("API Key exists")
else:
  raise ValueError("No Key found")

llm=ChatOpenAI(model="gpt-4o-mini",
               temperature=0.9)


API Key exists


In [11]:
imdb_prompt = ChatPromptTemplate.from_messages([
  ("system", "you are an IMDb movie reviewer"),
  ("human", "Write an IMDb-style review for the movie {movie}. Include a rating out of 10 and explain the strengths and weaknesses.")
])

rotten_tomatoes_prompt = ChatPromptTemplate.from_messages([
  ("system", "you are a Rotten Tomatoes movie reviewer"),
  ("human", "Write a Rotten Tomatoes-style review for the movie {movie}. Include a tomatometer-style score out of 100 and a concise critic-style opinion.")
])

In [12]:
llm = ChatOpenAI(model="gpt-4o-mini",
                 temperature=0.9)

In [13]:
str_parser = StrOutputParser()

In [14]:
imdb_chain = imdb_prompt | llm | str_parser

rotten_tomatoes_chain = rotten_tomatoes_prompt | llm | str_parser

In [15]:
parallel_reviews_chain = RunnableParallel(
  imdb=imdb_chain,
  rotten_tomatoes=rotten_tomatoes_chain,
 )

parallel_reviews_chain.invoke({"movie": "Inception"})

{'imdb': '**Title:** Inception  \n**Rating:** ★★★★★★★★☆ (8/10)  \n\n**Review:**  \nChristopher Nolan\'s *Inception* is a cinematic masterpiece that combines a compelling narrative with groundbreaking visual effects and a thought-provoking concept. Released in 2010, this science-fiction thriller takes viewers on a mind-bending journey through the world of dreams, challenging our perception of reality and illusion.\n\n**Strengths:**  \nOne of the film’s most significant strengths is its intricate plot structure. Nolan expertly weaves together multiple layers of reality, creating a narrative that requires the audience’s full attention. The concept of "dream sharing" is not only original but is explored in a unique and engaging manner. The screenplay is sharp, filled with clever dialogue and philosophical musings about the nature of dreams and the subconscious.\n\nThe cast delivers stellar performances, with Leonardo DiCaprio as Dom Cobb standing out as the haunted protagonist. The emotion

In [16]:
def beautify(response: dict) -> dict:
    return {
        "imdb": response["imdb"].strip(),
        "rotten_tomatoes": response["rotten_tomatoes"].strip(),
    }

beautify(parallel_reviews_chain.invoke({"movie": "Inception"}))

{'imdb': "**Title:** Inception  \n**Rating:** 9/10  \n\n**Review:**  \nChristopher Nolan’s *Inception* is a dazzling exploration of the mind's complexities that seamlessly blends science fiction with a heist thriller. Released in 2010, the film has left an indelible mark on cinematic storytelling, making viewers ponder the fine line between dreams and reality long after the credits roll.\n\n**Strengths:**  \nOne of the film’s most significant strengths is its innovative premise. The concept of navigating through layers of dreams to implant ideas is not only original but executed with remarkable precision. Nolan’s meticulous attention to detail ensures that each layer of the dream world is visually distinct yet logically interconnected, pulling the viewer deeper into its labyrinthine narrative.\n\nThe ensemble cast delivers standout performances, led by Leonardo DiCaprio as Dom Cobb, a skilled thief with a tortured past. His portrayal of a man grappling with guilt and loss adds emotiona

In [17]:
from pydantic import BaseModel
from typing import Literal

class MovieSummarySchema(BaseModel):
  movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm.with_structured_output(MovieSummarySchema)

In [18]:
result = llm_structured_output.invoke("A moving and well-acted sci-fi film with an ambitious story.")
result.model_dump

c:\Users\arunm\OneDrive\Desktop\LangChain_Tutorial\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MovieSummarySchema(movie_summary_flag='positive'), input_type=MovieSummarySchema])
  return self.__pydantic_serializer__.to_python(


<bound method BaseModel.model_dump of MovieSummarySchema(movie_summary_flag='positive')>